In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/test_p2.csv
/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/caja_negra_p2.py
/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/sample_submission_p2.csv
/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/train_p2.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('operacion-cazadores-de-bots-etapa-2-feature-eng')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng


In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

def evaluar_y_generar_submission(
    X_train,
    y_train,
    X_test,
    test_user_ids,
    output_path="submission_p2.csv"
):
    """
    Modelo de Caja Negra oficial para la Competencia Etapa 2.
    NO MODIFICAR este archivo. Su reto es mejorar el Feature Vector (X_train / X_test).
    """
    print("=" * 65)
    print("📦 MODELO CAJA NEGRA - ETAPA 2 (OPERACIÓN CAZADORES DE BOTS)")
    print("=" * 65)

    # Convertir a DataFrame si viene como ndarray para validar fácilmente
    X_tr = pd.DataFrame(X_train).copy()
    X_te = pd.DataFrame(X_test).copy()
    y_tr = pd.Series(y_train).values
    ids_te = pd.Series(test_user_ids).values

    # 1. Validaciones de dimensiones
    if len(X_tr) != 700 or len(y_tr) != 700:
        raise ValueError(f"❌ X_train y y_train deben tener exactamente 700 filas (tienen {len(X_tr)} y {len(y_tr)}).")
    if len(X_te) != 300 or len(ids_te) != 300:
        raise ValueError(f"❌ X_test y test_user_ids deben tener exactamente 300 filas (tienen {len(X_te)} y {len(ids_te)}).")
    if X_tr.shape[1] != X_te.shape[1]:
        raise ValueError(f"❌ X_train ({X_tr.shape[1]} cols) y X_test ({X_te.shape[1]} cols) deben tener las mismas columnas.")
    if list(X_tr.columns) != list(X_te.columns):
        raise ValueError("❌ El orden o nombre de las columnas en X_train y X_test no coincide.")

    # 2. Validación de tipos numéricos (sin strings sin codificar)
    non_num_tr = X_tr.select_dtypes(exclude=[np.number, bool]).columns.tolist()
    non_num_te = X_te.select_dtypes(exclude=[np.number, bool]).columns.tolist()
    if non_num_tr or non_num_te:
        raise TypeError(
            f"❌ La Caja Negra solo acepta variables numéricas. "
            f"Aún tienes texto sin codificar en: {sorted(list(set(non_num_tr + non_num_te)))}"
        )

    # 3. Validación de NaNs e Infinitos
    if X_tr.isna().any().any() or X_te.isna().any().any():
        raise ValueError("❌ Se detectaron valores NaN en X_train o X_test. Revisa tus transformaciones/ratios.")
    if np.isinf(X_tr.to_numpy(dtype=float)).any() or np.isinf(X_te.to_numpy(dtype=float)).any():
        raise ValueError("❌ Se detectaron valores infinitos (inf/-inf). ¿Dividiste entre cero en algún ratio?")

    # 4. Modelo fijo (semilla e hiperparámetros bloqueados)
    modelo = LogisticRegression(
        C=0.5,
        max_iter=300,
        random_state=42,
        class_weight="balanced"
    )

    # 5. Validación Cruzada Local (5-Fold Stratified)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring="f1_macro")
    mean_f1 = float(np.mean(scores))
    std_f1 = float(np.std(scores))

    # 6. Entrenamiento final y predicción sobre test_p2.csv
    modelo.fit(X_tr, y_tr)
    preds = modelo.predict(X_te).astype(int)

    df_sub = pd.DataFrame({
        "user_id": ids_te,
        "is_bot": preds
    })
    df_sub.to_csv(output_path, index=False)

    print(f"📐 Dimensiones del Feature Vector : {X_tr.shape[1]} features")
    print(f"🎯 CV Local (5-Fold Macro F1)     : {mean_f1:.4f} (+/- {std_f1:.4f})")
    print(f"🤖 Predicciones en Test           : {(preds == 1).sum()} bots | {(preds == 0).sum()} humanos")
    print(f"💾 Archivo generado               : {output_path}")
    if mean_f1 < 0.70:
        print("🟡 Sugerencia: Estás en zona Baseline. Revisa los encodings, escalamiento y pistas del acertijo.")
    elif mean_f1 < 0.85:
        print("🟠 Buen avance: Ya superaste el Baseline, pero aún hay señal oculta (ratios / clustering).")
    else:
        print("🟢 ¡Excelente Feature Engineering! Tu vector ya desbloqueó la señal de alto nivel.")
    print("=" * 65)
    return mean_f1, df_sub

In [4]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_train = pd.read_csv("/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/train_p2.csv")
df_test = pd.read_csv("/kaggle/input/competitions/operacion-cazadores-de-bots-etapa-2-feature-eng/test_p2.csv")

df_train.head()


,user_id,fecha_creacion,edad_usuario,pais,dispositivo,nivel_verificacion,seguidores,siguiendo,posts_totales,tasa_interaccion,hora_pico_actividad,email_domain,avg_session_min,variance_post_interval_hr,likes_given_30d,is_bot
0,USR_0590,2026-09-13,26,ARGENTINA,WEB,NINGUNO,360,363,466,0.3935,20,gmail.com,38.03,25.80,32,1
1,USR_0692,2025-10-24,25,ARGENTINA,ANDROID,TELEFONO,1937,3972,25,0.5257,9,yahoo.com,3.96,18.74,171,0
2,USR_0771,2025-05-27,30,ARGENTINA,IOS,CORREO,434,2028,232,0.0852,16,tempmail.xyz,23.25,7.47,57,0
3,USR_0832,2026-05-04,27,CHILE,ANDROID,VERIFICADO,1174,18,57,0.1267,11,gmail.com,9.50,15.15,78,0
4,USR_0415,2024-12-31,25,COLOMBIA,ANDROID,TELEFONO,780,117,56,0.1192,0,gmail.com,14.66,2.79,38,0


In [5]:
y_train = df_train["is_bot"].copy()

test_user_ids = df_test["user_id"].copy()

X_train = df_train.drop(columns=["is_bot", "user_id"]).copy()
X_test = df_test.drop(columns=["user_id"]).copy()

X_train.head()

baseline_train = X_train.select_dtypes(include="number")
baseline_test = X_test.select_dtypes(include="number")

cv_baseline, sub_baseline = evaluar_y_generar_submission(
    X_train=baseline_train,
    y_train=y_train,
    X_test=baseline_test,
    test_user_ids=test_user_ids,
    output_path="submission_baseline.csv"
)

📦 MODELO CAJA NEGRA - ETAPA 2 (OPERACIÓN CAZADORES DE BOTS)
📐 Dimensiones del Feature Vector : 9 features
🎯 CV Local (5-Fold Macro F1)     : 0.6420 (+/- 0.0225)
🤖 Predicciones en Test           : 131 bots | 169 humanos
💾 Archivo generado               : submission_baseline.csv
🟡 Sugerencia: Estás en zona Baseline. Revisa los encodings, escalamiento y pistas del acertijo.


In [6]:
# Pista 1 — La Escalera Oculta: nivel_verificacion tiene orden natural
ORDEN_VERIFICACION = {'NINGUNO': 0, 'CORREO': 1, 'TELEFONO': 2, 'VERIFICADO': 3}

# Pista 2 — Etiquetas sin jerarquía → One-Hot (aprendido SOLO en train)
COLS_NOMINALES = ['pais', 'dispositivo', 'email_domain']

# Pista 3 
FECHA_CORTE = pd.Timestamp("2026-11-25")

# Pista 4 — Conteos con colas largas
COLS_COLA_LARGA = ['seguidores', 'siguiendo', 'posts_totales', 'likes_given_30d', 'avg_session_min']

def features_base(df):
    X = pd.DataFrame(index=df.index)

    # Pista 1: Label Encoding
    X['verificacion_ord'] = df['nivel_verificacion'].map(ORDEN_VERIFICACION)
    
    # Pista 3: Reloj circular (23h queda pegada a 0h)
    X['hora_sin'] = np.sin(2 * np.pi * df['hora_pico_actividad'] / 24)
    X['hora_cos'] = np.cos(2 * np.pi * df['hora_pico_actividad'] / 24)

    # Pista 3: Fecha → días de vida de la cuenta
    dias = (FECHA_CORTE - pd.to_datetime(df['fecha_creacion'])).dt.days
    X['dias_vida_log'] = np.log1p(dias)

    # Pista 4: Comprimir colas largas
    for col in COLS_COLA_LARGA:
        X[f'{col}_log'] = np.log1p(df[col])

    # Variables que ya están en rango razonable
    for col in ['edad_usuario', 'tasa_interaccion', 'variance_post_interval_hr']:
        X[col] = df[col]

    
    
    return X

encoder_oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder_oh.fit(df_train[COLS_NOMINALES])

def features_onehot(df):
    return pd.DataFrame(encoder_oh.transform(df[COLS_NOMINALES]),
                        columns=encoder_oh.get_feature_names_out(), index=df.index)

print("Columnas One-Hot:", list(encoder_oh.get_feature_names_out()))

X_train = features_base(X_train)
X_test = features_base(X_test)

print(X_train.head())

Columnas One-Hot: ['pais_ARGENTINA', 'pais_CHILE', 'pais_COLOMBIA', 'pais_ESPANA', 'pais_MEXICO', 'dispositivo_ANDROID', 'dispositivo_API_SCRIPT', 'dispositivo_IOS', 'dispositivo_WEB', 'email_domain_gmail.com', 'email_domain_outlook.com', 'email_domain_proton.me', 'email_domain_tempmail.xyz', 'email_domain_yahoo.com']
   verificacion_ord  hora_sin  hora_cos  dias_vida_log  seguidores_log  \
0                 0 -0.866025  0.500000       4.304065        5.888878   
1                 2  0.707107 -0.707107       5.986452        7.569412   
2                 1 -0.866025 -0.500000       6.306275        6.075346   
3                 3  0.258819 -0.965926       5.327876        7.069023   
4                 2  0.000000  1.000000       6.543912        6.660575   

   siguiendo_log  posts_totales_log  likes_given_30d_log  avg_session_min_log  \
0       5.897154           6.146329             3.496508             3.664331   
1       8.287277           3.258097             5.147494             1.60

In [7]:
cv_encoding, sub_encoding = evaluar_y_generar_submission(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    test_user_ids=test_user_ids,
    output_path="submission_encoding.csv"
)

📦 MODELO CAJA NEGRA - ETAPA 2 (OPERACIÓN CAZADORES DE BOTS)
📐 Dimensiones del Feature Vector : 12 features
🎯 CV Local (5-Fold Macro F1)     : 0.7542 (+/- 0.0146)
🤖 Predicciones en Test           : 124 bots | 176 humanos
💾 Archivo generado               : submission_encoding.csv
🟠 Buen avance: Ya superaste el Baseline, pero aún hay señal oculta (ratios / clustering).
